# Partie B : Modèle de fondation temporel en zero-shot

## Question 7 : Prévision zero-shot avec Tiny Time Mixers (TTM)

### Objectif

Dans cette partie, on introduit un **Time Series Foundation Model (TSFM)** afin d'étudier une approche différente des modèles construits dans la Partie A.

Un TSFM est un modèle pré-entraîné sur un grand ensemble de séries temporelles. L'objectif du pré-entraînement est d'apprendre des structures générales présentes dans différents types de séries : dépendances temporelles, tendances, variations ou motifs récurrents.

Une fois pré-entraîné, le modèle peut être appliqué à une nouvelle série temporelle sans réentraîner ses paramètres. On parle alors de prévision **zero-shot**.

Dans cette question, on utilise un modèle **Tiny Time Mixers (TTM)** pré-entraîné de la bibliothèque `granite-tsfm`.

### Principe du zero-shot

Dans une approche classique supervisée, comme le Gradient Boosting de la Partie A, le modèle est entraîné directement sur les données d'entraînement du problème étudié.

En zero-shot, le principe est différent :

$$
\text{pré-entraînement sur de nombreuses séries}
\longrightarrow
\text{modèle pré-entraîné}
\longrightarrow
\text{application directe à une nouvelle série}
$$

Les paramètres du modèle TTM ne seront donc pas réentraînés sur notre série de production éolienne dans cette question.

À chaque origine de prévision, le modèle reçoit une fenêtre d'observations passées appelée **fenêtre de contexte** et produit une prévision pour l'horizon demandé.

### Tiny Time Mixers

Tiny Time Mixers est une famille de modèles de fondation temporels conçue pour effectuer notamment des prévisions zero-shot et few-shot.

TTM utilise une architecture compacte basée sur le mélange d'informations temporelles. Malgré sa taille relativement faible, le modèle est pré-entraîné pour apprendre des représentations temporelles réutilisables sur de nouvelles séries.

Dans cette question, on s'intéresse uniquement au mode **zero-shot** : les poids du modèle pré-entraîné restent inchangés.

### Protocole expérimental

Pour conserver une comparaison cohérente avec la Partie A, on utilise le même principe d'**origine glissante** sur la période de test.

À chaque origine \(t\) :

1. on fournit au modèle uniquement les observations disponibles avant \(t\) ;
2. on construit la fenêtre de contexte demandée par TTM ;
3. le modèle produit la prévision de l'observation suivante ;
4. l'origine est ensuite déplacée d'une heure ;
5. l'opération est répétée sur toute la période de test.

Aucune observation future ne doit être utilisée pour construire la prévision.

### Évaluation

Les prévisions TTM seront évaluées avec la **MASE (Mean Absolute Scaled Error)**, déjà utilisée dans la Partie A :

$$
MASE =
\frac{\frac{1}{n}\sum_{t=1}^{n}|y_t-\hat y_t|}
{\text{erreur absolue moyenne de référence calculée sur l'entraînement}}
$$

L'échelle utilisée doit rester cohérente avec celle définie précédemment afin que les résultats soient comparables.

Le MASE du TTM sera ensuite comparé à ceux obtenus pour :

- la baseline naïve ;
- le Gradient Boosting ;
- le modèle TTM en zero-shot.

L'objectif est de déterminer si un modèle pré-entraîné, utilisé sans adaptation à notre série, peut obtenir des performances compétitives par rapport aux modèles étudiés dans la Partie A.

### Point méthodologique important

Cette question évalue la capacité de **transfert zero-shot** du modèle TTM.

Si le modèle fournit de bonnes prévisions sans avoir été réentraîné sur la production éolienne étudiée, cela indique que les représentations apprises pendant son pré-entraînement peuvent être utiles sur une nouvelle série.

La comparaison doit néanmoins rester prudente : les modèles n'ont pas appris dans les mêmes conditions. Le Gradient Boosting est entraîné spécifiquement sur nos données, tandis que TTM utilise des paramètres appris auparavant sur d'autres séries.

In [3]:
# ============================================================
# Reprise des données et du protocole expérimental
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# Répertoire racine du projet
PROJECT_DIR = Path.cwd()

# Chemin du fichier de données
DATA_PATH = PROJECT_DIR / "data" / "raw" / "time_series_60min_singleindex.csv"

# Vérification de l'existence du fichier
print("Répertoire du projet :", PROJECT_DIR)
print("Fichier trouvé :", DATA_PATH.exists())

# Variable cible
TARGET = "DE_wind_onshore_generation_actual"

# Chargement uniquement des colonnes nécessaires pour Q7
data = pd.read_csv(
    DATA_PATH,
    usecols=["utc_timestamp", TARGET]
)

# Conversion du timestamp
data["utc_timestamp"] = pd.to_datetime(
    data["utc_timestamp"],
    utc=True
)

# Tri chronologique et passage du timestamp en index
data = (
    data
    .sort_values("utc_timestamp")
    .set_index("utc_timestamp")
)

print("\n--- Vérification des données ---")
print("Dimensions :", data.shape)
print("Début :", data.index.min())
print("Fin   :", data.index.max())
print("Valeurs manquantes cible :", data[TARGET].isna().sum())

Répertoire du projet : c:\Users\Said O\Desktop\Projets\TSFM_Wind_Forecasting
Fichier trouvé : True

--- Vérification des données ---
Dimensions : (50401, 1)
Début : 2014-12-31 23:00:00+00:00
Fin   : 2020-09-30 23:00:00+00:00
Valeurs manquantes cible : 73


In [4]:
# ============================================================
# Q7 — Sélection du modèle Tiny Time Mixers (TTM)
# ============================================================

from tsfm_public.toolkit.get_model import get_model

MODEL_PATH = "ibm-granite/granite-timeseries-ttm-r2"

# Notre série possède largement plus de 1536 observations historiques.
# On autorise donc la fonction à sélectionner un modèle utilisant
# jusqu'à 1536 observations de contexte.
AVAILABLE_CONTEXT = 1536

# Prévision à un pas : une heure dans notre série horaire.
FORECAST_HORIZON = 1

model_key = get_model(
    model_path=MODEL_PATH,
    context_length=AVAILABLE_CONTEXT,
    prediction_length=FORECAST_HORIZON,
    return_model_key=True,
)

print("Modèle source :", MODEL_PATH)
print("Contexte disponible fourni :", AVAILABLE_CONTEXT)
print("Horizon demandé :", FORECAST_HORIZON)
print("Variante TTM sélectionnée :", model_key)

INFO:p-5068:t-16796:get_model.py:get_model:Loading model from: ibm-granite/granite-timeseries-ttm-r2


Modèle source : ibm-granite/granite-timeseries-ttm-r2
Contexte disponible fourni : 1536
Horizon demandé : 1
Variante TTM sélectionnée : 1536-96-r2


In [5]:
# ============================================================
# Q7 — Chargement du modèle TTM pré-entraîné en zero-shot
# ============================================================

from time import perf_counter
from tsfm_public.toolkit.get_model import get_model

MODEL_PATH = "ibm-granite/granite-timeseries-ttm-r2"

debut = perf_counter()

ttm_model = get_model(
    model_path=MODEL_PATH,
    context_length=1536,
    prediction_length=1,
)

duree = perf_counter() - debut

print("Modèle TTM chargé avec succès.")
print(f"Durée du chargement : {duree:.2f} s")

print("\n--- Configuration du modèle ---")
print("Context length :", ttm_model.config.context_length)
print("Prediction length native :", ttm_model.config.prediction_length)

# Vérification du filtrage éventuel de l'horizon
print(
    "Prediction filter length :",
    getattr(ttm_model.config, "prediction_filter_length", None)
)

INFO:p-5068:t-16796:get_model.py:get_model:Loading model from: ibm-granite/granite-timeseries-ttm-r2
INFO:p-5068:t-16796:get_model.py:get_model:Model loaded successfully from ibm-granite/granite-timeseries-ttm-r2, revision = 1536-96-r2.
INFO:p-5068:t-16796:get_model.py:get_model:[TTM] context_length = 1536, prediction_length = 96


Modèle TTM chargé avec succès.
Durée du chargement : 1.36 s

--- Configuration du modèle ---
Context length : 1536
Prediction length native : 96
Prediction filter length : 1


In [6]:
# ============================================================
# Reconstruction du découpage temporel de la Partie A
# ============================================================

# Frontières obtenues dans la Partie A
train_end_time = pd.Timestamp("2019-01-09 22:00:00", tz="UTC")
validation_end_time = pd.Timestamp("2019-11-20 22:00:00", tz="UTC")

train = data.loc[data.index <= train_end_time].copy()

validation = data.loc[
    (data.index > train_end_time)
    & (data.index <= validation_end_time)
].copy()

test = data.loc[
    data.index > validation_end_time
].copy()

print("--- Découpage temporel ---")

print(
    f"Train      : {len(train):,} observations | "
    f"{train.index.min()} → {train.index.max()}"
)

print(
    f"Validation : {len(validation):,} observations | "
    f"{validation.index.min()} → {validation.index.max()}"
)

print(
    f"Test       : {len(test):,} observations | "
    f"{test.index.min()} → {test.index.max()}"
)

--- Découpage temporel ---
Train      : 35,280 observations | 2014-12-31 23:00:00+00:00 → 2019-01-09 22:00:00+00:00
Validation : 7,560 observations | 2019-01-09 23:00:00+00:00 → 2019-11-20 22:00:00+00:00
Test       : 7,561 observations | 2019-11-20 23:00:00+00:00 → 2020-09-30 23:00:00+00:00


In [9]:
# ============================================================
# Q7 — Construction de la première fenêtre de contexte TTM
# ============================================================

CONTEXT_LENGTH = ttm_model.config.context_length

# Première date de la période de test
first_test_time = test.index[0]

# Historique strictement antérieur à la première origine de test
historique_avant_test = data.loc[
    data.index < first_test_time,
    TARGET
]

# Les 1536 dernières observations disponibles avant le test
first_context = historique_avant_test.tail(CONTEXT_LENGTH)

print("Première origine de prévision :", first_test_time)
print("Longueur de contexte requise :", CONTEXT_LENGTH)
print("Longueur obtenue :", len(first_context))

print("\n--- Fenêtre de contexte ---")
print("Début :", first_context.index[0])
print("Fin   :", first_context.index[-1])

print("\nValeurs manquantes :", first_context.isna().sum())

# Vérification de la continuité horaire
ecarts = first_context.index.to_series().diff().dropna()

print(
    "Fréquence horaire régulière :",
    (ecarts == pd.Timedelta(hours=1)).all()
)

print("\nValeur réelle à prévoir :")
print(data.loc[first_test_time, TARGET])

Première origine de prévision : 2019-11-20 23:00:00+00:00
Longueur de contexte requise : 1536
Longueur obtenue : 1536

--- Fenêtre de contexte ---
Début : 2019-09-17 23:00:00+00:00
Fin   : 2019-11-20 22:00:00+00:00

Valeurs manquantes : 0
Fréquence horaire régulière : True

Valeur réelle à prévoir :
1993.0


In [10]:
# ============================================================
# Q7 — Première prévision zero-shot avec TTM
# ============================================================

import torch
import numpy as np
from time import perf_counter

# Transformation de la fenêtre en tableau numérique
context_values = first_context.to_numpy(
    dtype=np.float32
).reshape(1, CONTEXT_LENGTH, 1)

# Conversion en tenseur PyTorch
past_values = torch.tensor(
    context_values,
    dtype=torch.float32
)

print("Dimensions de l'entrée TTM :", past_values.shape)
print("Valeurs manquantes :", torch.isnan(past_values).sum().item())

# Mode évaluation : aucun entraînement
ttm_model.eval()

debut = perf_counter()

with torch.no_grad():
    outputs = ttm_model(
        past_values=past_values
    )

duree = perf_counter() - debut

# Prévisions produites par le modèle
prediction = outputs.prediction_outputs

print("\nDimensions de la sortie :", prediction.shape)
print(f"Durée de l'inférence : {duree:.4f} s")

print("\nSortie brute TTM :")
print(prediction)

Dimensions de l'entrée TTM : torch.Size([1, 1536, 1])
Valeurs manquantes : 0

Dimensions de la sortie : torch.Size([1, 1, 1])
Durée de l'inférence : 0.0134 s

Sortie brute TTM :
tensor([[[2679.7207]]])


In [11]:
# ============================================================
# Q7 — Vérification de la première prévision zero-shot
# ============================================================

# Extraction de la prévision scalaire
first_ttm_pred = prediction[0, 0, 0].item()

# Valeur réellement observée à la première origine du test
first_actual = data.loc[first_test_time, TARGET]

# Erreur absolue
first_abs_error = abs(first_actual - first_ttm_pred)

# Dernière observation connue avant la prévision
last_observed = first_context.iloc[-1]

print("Date prévue :", first_test_time)

print(f"\nDernière observation connue : {last_observed:.2f} MW")
print(f"Prévision TTM              : {first_ttm_pred:.2f} MW")
print(f"Valeur réellement observée : {first_actual:.2f} MW")
print(f"Erreur absolue              : {first_abs_error:.2f} MW")

Date prévue : 2019-11-20 23:00:00+00:00

Dernière observation connue : 1908.00 MW
Prévision TTM              : 2679.72 MW
Valeur réellement observée : 1993.00 MW
Erreur absolue              : 686.72 MW


In [12]:
# ============================================================
# Q7 — Construction des fenêtres à origine glissante
# ============================================================

from time import perf_counter
import numpy as np

debut = perf_counter()

# Série complète sous forme numérique
y = data[TARGET].to_numpy(dtype=np.float32)

# Position de la première observation du test dans la série complète
first_test_pos = data.index.get_loc(test.index[0])

contexts = []
targets = []
timestamps = []
valid_positions = []

for pos in range(first_test_pos, len(data)):
    
    # Il faut disposer de 1536 observations antérieures
    if pos < CONTEXT_LENGTH:
        continue

    context = y[pos - CONTEXT_LENGTH:pos]
    target = y[pos]

    # On conserve uniquement les origines pour lesquelles
    # le contexte ET la cible sont observés
    if np.isnan(context).any() or np.isnan(target):
        continue

    contexts.append(context)
    targets.append(target)
    timestamps.append(data.index[pos])
    valid_positions.append(pos)

contexts = np.asarray(contexts, dtype=np.float32)
targets = np.asarray(targets, dtype=np.float32)
timestamps = pd.DatetimeIndex(timestamps)

# Ajout de la dimension "variable"
contexts = contexts[..., np.newaxis]

duree = perf_counter() - debut

print("Nombre total d'origines du test :", len(test))
print("Origines utilisables :", len(contexts))
print("Origines exclues :", len(test) - len(contexts))

print("\nDimensions des contextes :", contexts.shape)
print("Dimensions des cibles :", targets.shape)

print("\nPremière origine :", timestamps[0])
print("Dernière origine :", timestamps[-1])

print("\nNaN dans les contextes :", np.isnan(contexts).sum())
print("NaN dans les cibles :", np.isnan(targets).sum())

print(f"\nDurée de préparation : {duree:.2f} s")

Nombre total d'origines du test : 7561
Origines utilisables : 7561
Origines exclues : 0

Dimensions des contextes : (7561, 1536, 1)
Dimensions des cibles : (7561,)

Première origine : 2019-11-20 23:00:00+00:00
Dernière origine : 2020-09-30 23:00:00+00:00

NaN dans les contextes : 0
NaN dans les cibles : 0

Durée de préparation : 0.14 s


In [13]:
# ============================================================
# Q7 — Inférence zero-shot TTM sur toute la période de test
# ============================================================

import torch
import numpy as np
from time import perf_counter

BATCH_SIZE = 32

ttm_model.eval()

ttm_predictions = []

debut = perf_counter()

with torch.no_grad():

    for start in range(0, len(contexts), BATCH_SIZE):

        end = min(start + BATCH_SIZE, len(contexts))

        # Conversion du batch NumPy en tenseur PyTorch
        batch_values = torch.from_numpy(
            contexts[start:end]
        )

        # Inférence zero-shot
        outputs = ttm_model(
            past_values=batch_values
        )

        # Sortie attendue :
        # (batch, horizon=1, variable=1)
        batch_predictions = (
            outputs.prediction_outputs[:, 0, 0]
            .cpu()
            .numpy()
        )

        ttm_predictions.append(batch_predictions)

ttm_predictions = np.concatenate(ttm_predictions)

duree = perf_counter() - debut

print("--- Inférence TTM terminée ---")
print(f"Nombre de prévisions : {len(ttm_predictions)}")
print(f"Batch size : {BATCH_SIZE}")
print(f"Durée totale : {duree:.2f} s")
print(f"Durée moyenne par origine : {duree / len(ttm_predictions):.4f} s")

print("\n--- Contrôles ---")
print("Dimensions :", ttm_predictions.shape)
print("NaN :", np.isnan(ttm_predictions).sum())
print("Valeur minimale :", ttm_predictions.min())
print("Valeur maximale :", ttm_predictions.max())

print("\nPremière prévision :", ttm_predictions[0])
print("Première valeur réelle :", targets[0])

--- Inférence TTM terminée ---
Nombre de prévisions : 7561
Batch size : 32
Durée totale : 8.59 s
Durée moyenne par origine : 0.0011 s

--- Contrôles ---
Dimensions : (7561,)
NaN : 0
Valeur minimale : -86.97656
Valeur maximale : 40161.008

Première prévision : 2679.7207
Première valeur réelle : 1993.0


In [14]:
# ======================================================================
# Q7 — Évaluation ponctuelle du TTM sur la période de test (MAE, MASE)
# ======================================================================

# Erreurs absolues du TTM
ttm_abs_errors = np.abs(targets - ttm_predictions)

# MAE du TTM sur le test
ttm_mae = ttm_abs_errors.mean()

# ------------------------------------------------------------
# Échelle saisonnière de la MASE calculée uniquement sur le train
# saisonnalité horaire journalière : m = 24
# ------------------------------------------------------------

train_values = train[TARGET].to_numpy(dtype=np.float64)

train_current = train_values[24:]
train_lag24 = train_values[:-24]

# On ne conserve que les paires pour lesquelles
# les deux observations sont disponibles
valid_scale = (
    ~np.isnan(train_current)
    & ~np.isnan(train_lag24)
)

mase_scale = np.mean(
    np.abs(
        train_current[valid_scale]
        - train_lag24[valid_scale]
    )
)

# MASE du TTM
ttm_mase = ttm_mae / mase_scale

print("--- Évaluation TTM zero-shot sur le test ---")
print(f"Nombre de prévisions : {len(ttm_predictions)}")

print(f"\nMAE TTM : {ttm_mae:.2f} MW")
print(f"Échelle saisonnière train : {mase_scale:.2f} MW")
print(f"MASE TTM : {ttm_mase:.3f}")

--- Évaluation TTM zero-shot sur le test ---
Nombre de prévisions : 7561

MAE TTM : 939.88 MW
Échelle saisonnière train : 4996.19 MW
MASE TTM : 0.188


In [15]:
# ============================================================
# Q7 — Reconstruction des variables du Gradient Boosting de Q4
# ============================================================

# Copie pour ne pas modifier directement les données originales
data_gb = data.copy()

# ------------------------------------------------------------
# 1. Variables calendaires
# ------------------------------------------------------------

data_gb["hour"] = data_gb.index.hour
data_gb["dayofweek"] = data_gb.index.dayofweek
data_gb["month"] = data_gb.index.month

# ------------------------------------------------------------
# 2. Variables cycliques
# ------------------------------------------------------------

data_gb["hour_sin"] = np.sin(
    2 * np.pi * data_gb["hour"] / 24
)

data_gb["hour_cos"] = np.cos(
    2 * np.pi * data_gb["hour"] / 24
)

data_gb["day_sin"] = np.sin(
    2 * np.pi * data_gb["dayofweek"] / 7
)

data_gb["day_cos"] = np.cos(
    2 * np.pi * data_gb["dayofweek"] / 7
)

# ------------------------------------------------------------
# 3. Retards temporels identiques à Q4
# ------------------------------------------------------------

LAGS = [1, 2, 3, 24, 48, 168]

for lag in LAGS:
    data_gb[f"lag_{lag}"] = data_gb[TARGET].shift(lag)

# ------------------------------------------------------------
# 4. Variation de la production
# ------------------------------------------------------------

data_gb["delta_target"] = (
    data_gb[TARGET] - data_gb[TARGET].shift(1)
)

# ------------------------------------------------------------
# 5. Liste exacte des variables explicatives
# ------------------------------------------------------------

FEATURES = [
    "hour",
    "dayofweek",
    "month",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_24",
    "lag_48",
    "lag_168",
]

print("Nombre de variables explicatives :", len(FEATURES))

print("\nVariables utilisées :")
for feature in FEATURES:
    print("-", feature)

print("\nDimensions de data_gb :", data_gb.shape)

Nombre de variables explicatives : 13

Variables utilisées :
- hour
- dayofweek
- month
- hour_sin
- hour_cos
- day_sin
- day_cos
- lag_1
- lag_2
- lag_3
- lag_24
- lag_48
- lag_168

Dimensions de data_gb : (50401, 15)


In [16]:
# ================================================================
# Q7 — Préparation du train et du test pour le Gradient Boosting
# ================================================================

# Frontières temporelles identiques à la Partie A
train_end_time = pd.Timestamp("2019-01-09 22:00:00", tz="UTC")
validation_end_time = pd.Timestamp("2019-11-20 22:00:00", tz="UTC")

# ------------------------------------------------------------
# 1. Séparation chronologique
# ------------------------------------------------------------

train_gb_q7 = data_gb.loc[
    data_gb.index <= train_end_time
].copy()

test_gb_q7 = data_gb.loc[
    data_gb.index > validation_end_time
].copy()

print("--- Avant suppression des valeurs manquantes ---")
print("Train :", len(train_gb_q7))
print("Test  :", len(test_gb_q7))

# ------------------------------------------------------------
# 2. Suppression uniquement des lignes inutilisables
# ------------------------------------------------------------

colonnes_requises = FEATURES + ["delta_target", TARGET]

train_gb_q7 = train_gb_q7.dropna(
    subset=colonnes_requises
)

test_gb_q7 = test_gb_q7.dropna(
    subset=colonnes_requises
)

# ------------------------------------------------------------
# 3. Construction des matrices
# ------------------------------------------------------------

X_train_gb = train_gb_q7[FEATURES]
y_train_delta = train_gb_q7["delta_target"]

X_test_gb = test_gb_q7[FEATURES]
y_test_gb = test_gb_q7[TARGET]

# ------------------------------------------------------------
# 4. Vérifications
# ------------------------------------------------------------

print("\n--- Après suppression des valeurs manquantes ---")
print("Train utilisable :", len(X_train_gb))
print("Test utilisable  :", len(X_test_gb))

print("\nDimensions :")
print("X_train :", X_train_gb.shape)
print("X_test  :", X_test_gb.shape)

print("\nPériode du test :")
print("Première date :", test_gb_q7.index.min())
print("Dernière date :", test_gb_q7.index.max())

print("\nValeurs manquantes :")
print("X_train :", X_train_gb.isna().sum().sum())
print("X_test  :", X_test_gb.isna().sum().sum())

--- Avant suppression des valeurs manquantes ---
Train : 35280
Test  : 7561

--- Après suppression des valeurs manquantes ---
Train utilisable : 34823
Test utilisable  : 7561

Dimensions :
X_train : (34823, 13)
X_test  : (7561, 13)

Période du test :
Première date : 2019-11-20 23:00:00+00:00
Dernière date : 2020-09-30 23:00:00+00:00

Valeurs manquantes :
X_train : 0
X_test  : 0


In [17]:
# ============================================================
# Q7 — Gradient Boosting de Q4 : évaluation finale sur le test
# ============================================================

from sklearn.ensemble import GradientBoostingRegressor
from time import perf_counter
import numpy as np

# ------------------------------------------------------------
# 1. Même modèle que celui retenu en Q4
# ------------------------------------------------------------

gb_delta_q7 = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    loss="squared_error",
    random_state=42
)

# ------------------------------------------------------------
# 2. Entraînement sur le train uniquement
# ------------------------------------------------------------

debut = perf_counter()

gb_delta_q7.fit(
    X_train_gb,
    y_train_delta
)

# ------------------------------------------------------------
# 3. Prévision des variations sur le test
# ------------------------------------------------------------

delta_pred_test = gb_delta_q7.predict(X_test_gb)

# ---------------------------------------------------------------------
# 4. Reconstruction de la production : y_hat(t) = y(t-1) + delta_hat(t)
# ---------------------------------------------------------------------

gb_predictions_test = (
    test_gb_q7["lag_1"].to_numpy()
    + delta_pred_test
)

# ------------------------------------------------------------
# 5. MAE sur le test
# ------------------------------------------------------------

gb_abs_errors = np.abs(
    y_test_gb.to_numpy() - gb_predictions_test
)

gb_mae_test = gb_abs_errors.mean()

# ------------------------------------------------------------
# 6. MASE avec exactement la même échelle que Q3 et TTM
# ------------------------------------------------------------

gb_mase_test = gb_mae_test / mase_scale

duree = perf_counter() - debut

# ------------------------------------------------------------
# 7. Résultats
# ------------------------------------------------------------

print("--- Gradient Boosting en variation : test ---")
print(f"Nombre de prévisions : {len(gb_predictions_test)}")
print(f"MAE GB : {gb_mae_test:.2f} MW")
print(f"Échelle saisonnière train : {mase_scale:.2f} MW")
print(f"MASE GB : {gb_mase_test:.3f}")
print(f"Durée totale : {duree:.2f} s")

--- Gradient Boosting en variation : test ---
Nombre de prévisions : 7561
MAE GB : 599.75 MW
Échelle saisonnière train : 4996.19 MW
MASE GB : 0.120
Durée totale : 7.82 s


## Q7 : TSFM et Tiny Time Mixers (TTM) en zero-shot

### Principe d'un TSFM

Un **Time Series Foundation Model (TSFM)** est un modèle pré-entraîné sur un grand ensemble de séries temporelles provenant de domaines variés. L'objectif est d'apprendre des structures temporelles générales qui peuvent ensuite être réutilisées sur de nouvelles séries.

Dans cette question, on utilise **Tiny Time Mixers (TTM)** en **zero-shot**. Cela signifie qu'on applique directement le modèle pré-entraîné à notre série de production éolienne, **sans réentraîner ses paramètres sur notre ensemble d'entraînement**.

Il faut donc bien distinguer :

- le **pré-entraînement**, réalisé auparavant sur un grand corpus de séries temporelles ;
- le **zero-shot**, qui consiste à utiliser le modèle pré-entraîné sur notre série sans nouvelle phase d'apprentissage.

À chaque origine de prévision, on fournit tout de même au modèle une fenêtre contenant les observations passées de notre série. Cette fenêtre constitue le **contexte d'entrée** du modèle et non un nouvel entraînement.

### Configuration de TTM

On utilise le modèle pré-entraîné :

`ibm-granite/granite-timeseries-ttm-r2`

La variante automatiquement sélectionnée est :

`1536-96-r2`

Le modèle utilise un contexte de **1536 observations**. Comme notre série est horaire, cela représente :

$$
\frac{1536}{24}=64 \text{ jours}.
$$

Pour chaque origine de prévision, TTM dispose donc des **64 derniers jours de production observée**.

La longueur native de prévision de la variante chargée est de 96 pas, mais on filtre la sortie pour conserver uniquement le **premier pas**, puisque notre objectif est de prévoir la production **une heure à l'avance**.

Le contexte de 1536 observations n'a pas été choisi en comparant plusieurs configurations sur le jeu de test. On utilise une longueur de contexte prédéfinie supportée par la variante TTM-R2 retenue. Cela permet de ne pas utiliser le test pour régler le modèle.

### Protocole d'origine glissante

On applique TTM sur toute la période de test avec une **origine glissante**.

Pour chaque heure \(t\) :

1. on récupère les 1536 observations précédant \(t\) ;
2. on fournit cette fenêtre à TTM ;
3. on prédit la production à l'instant \(t\) ;
4. on avance ensuite d'une heure et on recommence.

Ainsi, à chaque origine de prévision, on utilise uniquement les informations disponibles dans le passé.

Toutes les observations du test sont exploitables :

$$
\boxed{7561\text{ prévisions}}
$$

sur la période allant du **20 novembre 2019 à 23:00 UTC** au **30 septembre 2020 à 23:00 UTC**.

Pour accélérer les calculs, on regroupe les fenêtres par batch lors de l'inférence. Cela ne modifie pas le protocole temporel : chaque fenêtre reste construite uniquement à partir des observations antérieures à la date prédite.

### Évaluation avec la MASE

Pour comparer les modèles sur une même échelle, on utilise la **MASE** (*Mean Absolute Scaled Error*) :

$$
MASE =
\frac{MAE_{\text{modèle,test}}}
{\text{erreur saisonnière moyenne calculée sur le train}}.
$$

Comme dans la question 3, l'échelle saisonnière est calculée sur le train à partir d'un retard de 24 heures :

$$
\boxed{\text{Échelle saisonnière}=4996{,}19\text{ MW}}.
$$

On conserve exactement cette même échelle pour la naïve, le Gradient Boosting et TTM.

### Résultat de TTM en zero-shot

Sur les 7561 observations du test, on obtient :

$$
\boxed{MAE_{TTM}=939{,}88\text{ MW}}
$$
et :
$$
\boxed{MASE_{TTM}=0{,}188}.
$$

Une MASE de 0,188 signifie que la MAE de TTM représente environ **18,8 % de l'échelle saisonnière utilisée comme référence**.

Ce résultat est particulièrement intéressant car on l'obtient **sans réentraîner TTM sur notre ensemble d'apprentissage éolien**.

### Comparaison avec la naïve et le Gradient Boosting

Pour effectuer une comparaison cohérente, on évalue également le **Gradient Boosting en variation retenu en Q4** sur exactement les mêmes 7561 observations du test.

On conserve :

- les mêmes 13 variables explicatives ;
- les mêmes hyperparamètres ;
- les mêmes 34 823 observations d'entraînement utilisables ;
- la même formulation en variation :

$$
\Delta y_t=y_t-y_{t-1},
$$

avec reconstruction :

$$
\widehat{y}_t=y_{t-1}+\widehat{\Delta y}_t.
$$

On obtient finalement :

| Modèle | MAE sur le test (MW) | MASE sur le test |
|---|---:|---:|
| Naïve saisonnière $t-24$ | 6100,71 | 1,221 |
| **Gradient Boosting en variation** | **599,75** | **0,120** |
| TTM zero-shot | 939,88 | 0,188 |

Le classement obtenu sur le test est donc :

$$
\boxed{\text{Gradient Boosting} < \text{TTM zero-shot} < \text{Naïve saisonnière}}
$$

où une valeur plus faible correspond à une meilleure performance.

Le **Gradient Boosting est le plus précis** sur cette période. Sa MAE est environ **36,2 % plus faible** que celle de TTM.

Cependant, les deux modèles ne reposent pas sur la même logique. Le Gradient Boosting est **entraîné spécifiquement sur notre série éolienne** et exploite notamment plusieurs valeurs retardées de cette série. TTM, au contraire, est utilisé directement en **zero-shot** à partir de ses paramètres pré-entraînés.

Le résultat de TTM reste donc intéressant : sans apprentissage spécifique sur notre train, il atteint une MASE de seulement **0,188** et améliore très nettement la référence naïve saisonnière.

### Limites et précautions

Cette comparaison doit être interprétée avec quelques précautions.

D'abord, on n'a testé qu'une configuration TTM dans cette évaluation. On n'a volontairement pas essayé plusieurs longueurs de contexte sur le test pour retenir celle donnant le meilleur résultat, car cela aurait utilisé le test pour régler le modèle.

Ensuite, TTM produit quelques prévisions légèrement négatives. La valeur minimale observée est d'environ **-87 MW**, ce qui n'est pas physiquement cohérent pour une production éolienne. On conserve néanmoins les prévisions brutes dans l'évaluation principale et on n'applique pas de correction à zéro après avoir observé le test, afin de ne pas modifier le protocole a posteriori.

Enfin, le résultat concerne **une seule série de production éolienne et un horizon d'une heure**. On ne peut donc pas en conclure que le Gradient Boosting sera systématiquement meilleur que TTM sur d'autres séries, d'autres sources d'énergie ou d'autres horizons de prévision.

### Conclusion

Cette expérience montre qu'un modèle spécialisé comme le Gradient Boosting reste ici le plus performant :

$$
MASE_{GB}=0{,}120
$$

contre :

$$
MASE_{TTM}=0{,}188.
$$

Cependant, TTM atteint cette performance **sans réentraînement sur notre série**, ce qui illustre l'intérêt des modèles de fondation pour séries temporelles : ils peuvent transférer des connaissances apprises lors du pré-entraînement vers une nouvelle tâche et fournir immédiatement une prévision compétitive.

Dans notre cas, TTM ne remplace donc pas le modèle spécialisé, mais constitue une **baseline pré-entraînée forte en zero-shot**.

### Références

**Référence scientifique**

Ekambaram, V., Jati, A., Dayama, P., Mukherjee, S., Nguyen, N. H., Gifford, W. M., Reddy, C. K., & Kalagnanam, J. (2024). *Tiny Time Mixers (TTMs): Fast Pre-trained Models for Enhanced Zero/Few-Shot Forecasting of Multivariate Time Series*. Advances in Neural Information Processing Systems (NeurIPS 2024).

**Documentation officielle**

- IBM Granite - *Granite TSFM*, dépôt officiel GitHub.
- IBM Granite - *Granite Time Series TTM-R2*, model card.
- IBM Granite - documentation de `get_model()` pour la sélection et le chargement des variantes TTM.

**Remarque :** le choix du protocole d'origine glissante, l'utilisation d'une MASE commune et la décision de ne pas modifier les prévisions négatives après observation du test correspondent à des **choix méthodologiques de ce projet** et ne sont pas présentés comme des recommandations imposées par la documentation TTM.

## Question 8 : Chronos-Bolt en zero-shot

Dans cette question, on répète l'expérience de la Q7 avec un second modèle de fondation pour séries temporelles : **Chronos-Bolt**, utilisé avec la bibliothèque `chronos-forecasting`.

Comme pour TTM, on utilise Chronos-Bolt en **zero-shot** : le modèle est déjà pré-entraîné et on ne réalise aucun réentraînement sur notre série de production éolienne.

L'objectif est de déterminer si les deux TSFM, **TTM et Chronos-Bolt**, présentent des performances comparables sur notre série.

### Protocole

Pour conserver une comparaison cohérente avec la Q7, on suit le même principe général :

1. on charge un modèle Chronos-Bolt pré-entraîné sans modifier ses paramètres ;
2. pour chaque origine du jeu de test, on construit un contexte contenant uniquement les observations disponibles avant la date à prévoir ;
3. on réalise une prévision à **un pas**, soit une heure à l'avance ;
4. on récupère les prévisions sous forme de **quantiles**, notamment les quantiles $q_{0.1}$, $q_{0.5}$ et $q_{0.9}$ ;
5. on utilise le quantile médian $q_{0.5}$ comme prévision ponctuelle pour calculer la **MAE** puis la **MASE** ;
6. on conserve la même échelle saisonnière calculée sur le train en Q3 :

$$
\text{échelle MASE}=4996{,}19\text{ MW}.
$$

On compare ensuite Chronos-Bolt aux résultats déjà obtenus sur le test :

| Modèle | MASE test |
|---|---:|
| Naïve saisonnière $t-24$ | 1,221 |
| Gradient Boosting en variation | 0,120 |
| TTM zero-shot | 0,188 |
| Chronos-Bolt zero-shot | à calculer |

On examinera également les quantiles prédictifs afin de compléter l'évaluation ponctuelle par une lecture probabiliste des prévisions.

### Principe méthodologique

Le modèle et son protocole sont fixés **avant l'évaluation sur le test**. On ne cherchera donc pas plusieurs configurations sur le test pour retenir celle donnant le meilleur score.

Si Chronos-Bolt ne fait pas mieux que la naïve saisonnière, on le signalera explicitement. L'objectif n'est pas d'obtenir artificiellement le meilleur résultat possible, mais de mesurer correctement la capacité de transfert du modèle en zero-shot sur notre série.

In [18]:
# Q8 — Vérification de l'environnement Chronos-Bolt

import torch
import chronos

from chronos import BaseChronosPipeline

print("PyTorch :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())
print("Import chronos : OK")
print("BaseChronosPipeline : OK")

PyTorch : 2.15.0+cpu
CUDA disponible : False
Import chronos : OK
BaseChronosPipeline : OK


In [19]:
# ============================================================
# Q8 — Chargement du modèle Chronos-Bolt
# ============================================================

from time import perf_counter
import torch
from chronos import BaseChronosPipeline

MODEL_CHRONOS = "amazon/chronos-bolt-base"

debut = perf_counter()

chronos_pipeline = BaseChronosPipeline.from_pretrained(
    MODEL_CHRONOS,
    device_map="cpu",
    dtype=torch.float32,
)

duree = perf_counter() - debut

print("--- Chronos-Bolt chargé avec succès ---")
print("Modèle :", MODEL_CHRONOS)
print("Device : CPU")
print("dtype :", torch.float32)
print(f"Durée de chargement : {duree:.2f} s")

config.json: 0.00B [00:00, ?B/s]

c:\Users\Said O\Desktop\Projets\TSFM_Wind_Forecasting\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Said O\.cache\huggingface\hub\models--amazon--chronos-bolt-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/821M [00:00<?, ?B/s]

--- Chronos-Bolt chargé avec succès ---
Modèle : amazon/chronos-bolt-base
Device : CPU
dtype : torch.float32
Durée de chargement : 37.36 s


In [20]:
# ============================================================
# Q8 — Première prévision Chronos-Bolt sur une seule origine
# ============================================================

from time import perf_counter
import torch

# Première date du test
first_test_time = test.index[0]

# Historique strictement antérieur à la première date du test
historique_chronos = data.loc[
    data.index < first_test_time,
    TARGET
]

# Contexte maximal retenu pour Chronos-Bolt
CHRONOS_CONTEXT_LENGTH = 2048

first_context_chronos = (
    historique_chronos
    .tail(CHRONOS_CONTEXT_LENGTH)
    .to_numpy(dtype="float32")
)

print("--- Vérification du contexte ---")
print("Date à prévoir :", first_test_time)
print("Nombre d'observations :", len(first_context_chronos))
print(
    "Première date du contexte :",
    historique_chronos.tail(CHRONOS_CONTEXT_LENGTH).index[0]
)
print(
    "Dernière date du contexte :",
    historique_chronos.tail(CHRONOS_CONTEXT_LENGTH).index[-1]
)
print(
    "Nombre de NaN :",
    int(torch.isnan(torch.tensor(first_context_chronos)).sum())
)

# Conversion en tenseur
context_tensor = torch.tensor(first_context_chronos)

# Quantiles demandés
QUANTILE_LEVELS = [0.1, 0.5, 0.9]

# ------------------------------------------------------------
# Prévision à un pas
# ------------------------------------------------------------

debut = perf_counter()

quantiles, mean = chronos_pipeline.predict_quantiles(
    inputs=context_tensor,
    prediction_length=1,
    quantile_levels=QUANTILE_LEVELS,
)

duree = perf_counter() - debut

print("\n--- Sortie brute Chronos-Bolt ---")
print("Type quantiles :", type(quantiles))
print("Dimensions quantiles :", quantiles.shape)

print("Type mean :", type(mean))
print("Dimensions mean :", mean.shape)

print("\nQuantiles bruts :")
print(quantiles)

print("\nPrévision moyenne :")
print(mean)

print(f"\nDurée d'inférence : {duree:.4f} s")

--- Vérification du contexte ---
Date à prévoir : 2019-11-20 23:00:00+00:00
Nombre d'observations : 2048
Première date du contexte : 2019-08-27 15:00:00+00:00
Dernière date du contexte : 2019-11-20 22:00:00+00:00
Nombre de NaN : 0


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.



--- Sortie brute Chronos-Bolt ---
Type quantiles : <class 'torch.Tensor'>
Dimensions quantiles : torch.Size([1, 1, 3])
Type mean : <class 'torch.Tensor'>
Dimensions mean : torch.Size([1, 1])

Quantiles bruts :
tensor([[[1148.0322, 1884.7656, 2344.3779]]])

Prévision moyenne :
tensor([[1884.7656]])

Durée d'inférence : 0.5387 s


In [21]:
# ============================================================
# Q8 — Construction des contextes Chronos-Bolt
#      pour toutes les origines du test
# ============================================================

from time import perf_counter
import numpy as np

debut = perf_counter()

# Série complète sous forme de tableau
y_chronos = data[TARGET].to_numpy(dtype=np.float32)

# Position de la première observation du test
first_test_pos = data.index.get_loc(test.index[0])

chronos_contexts = []
chronos_targets = []
chronos_timestamps = []

# ------------------------------------------------------------
# Construction des fenêtres en origine glissante
# ------------------------------------------------------------

for pos in range(first_test_pos, len(data)):

    # Les 2048 observations strictement antérieures à la cible
    context = y_chronos[
        pos - CHRONOS_CONTEXT_LENGTH : pos
    ]

    target = y_chronos[pos]

    # On conserve uniquement les origines entièrement exploitables
    if (
        len(context) == CHRONOS_CONTEXT_LENGTH
        and not np.isnan(context).any()
        and not np.isnan(target)
    ):
        chronos_contexts.append(context)
        chronos_targets.append(target)
        chronos_timestamps.append(data.index[pos])

# Conversion en tableaux NumPy
chronos_contexts = np.asarray(
    chronos_contexts,
    dtype=np.float32
)

chronos_targets = np.asarray(
    chronos_targets,
    dtype=np.float32
)

chronos_timestamps = np.asarray(
    chronos_timestamps
)

duree = perf_counter() - debut

# ------------------------------------------------------------
# Vérifications
# ------------------------------------------------------------

print("--- Contextes Chronos-Bolt ---")
print("Nombre total d'origines du test :", len(test))
print("Nombre d'origines exploitables :", len(chronos_contexts))
print("Nombre d'origines exclues :", len(test) - len(chronos_contexts))

print("\nDimensions des contextes :", chronos_contexts.shape)
print("Dimensions des cibles :", chronos_targets.shape)

print("\nPremière date :", chronos_timestamps[0])
print("Dernière date :", chronos_timestamps[-1])

print("\nNombre de NaN dans les contextes :", np.isnan(chronos_contexts).sum())
print("Nombre de NaN dans les cibles :", np.isnan(chronos_targets).sum())

#print(f"\nDurée de préparation : {duree:.2f} s")

--- Contextes Chronos-Bolt ---
Nombre total d'origines du test : 7561
Nombre d'origines exploitables : 7561
Nombre d'origines exclues : 0

Dimensions des contextes : (7561, 2048)
Dimensions des cibles : (7561,)

Première date : 2019-11-20 23:00:00+00:00
Dernière date : 2020-09-30 23:00:00+00:00

Nombre de NaN dans les contextes : 0
Nombre de NaN dans les cibles : 0


In [22]:
# ============================================================
# Q8 — Inférence Chronos-Bolt sur toute la période de test
# ============================================================

from time import perf_counter
import numpy as np
import torch

BATCH_SIZE_CHRONOS = 32
QUANTILE_LEVELS = [0.1, 0.5, 0.9]

chronos_quantiles_batches = []
chronos_mean_batches = []

debut = perf_counter()

# ------------------------------------------------------------
# Inférence par batch
# ------------------------------------------------------------

for start in range(0, len(chronos_contexts), BATCH_SIZE_CHRONOS):

    end = min(
        start + BATCH_SIZE_CHRONOS,
        len(chronos_contexts)
    )

    # Conversion du batch NumPy en tenseur PyTorch
    batch_context = torch.from_numpy(
        chronos_contexts[start:end]
    )

    # Prévisions par quantiles
    with torch.no_grad():

        batch_quantiles, batch_mean = (
            chronos_pipeline.predict_quantiles(
                inputs=batch_context,
                prediction_length=1,
                quantile_levels=QUANTILE_LEVELS,
            )
        )

    # Retour sur CPU puis conversion en NumPy
    chronos_quantiles_batches.append(
        batch_quantiles.cpu().numpy()
    )

    chronos_mean_batches.append(
        batch_mean.cpu().numpy()
    )

# ------------------------------------------------------------
# Regroupement de tous les batches
# ------------------------------------------------------------

chronos_quantiles = np.concatenate(
    chronos_quantiles_batches,
    axis=0
)

chronos_mean = np.concatenate(
    chronos_mean_batches,
    axis=0
)

duree = perf_counter() - debut

# ------------------------------------------------------------
# Extraction des trois quantiles
# Dimensions attendues : (7561, 1, 3)
# ------------------------------------------------------------

chronos_q10 = chronos_quantiles[:, 0, 0]
chronos_q50 = chronos_quantiles[:, 0, 1]
chronos_q90 = chronos_quantiles[:, 0, 2]

# ------------------------------------------------------------
# Vérifications
# ------------------------------------------------------------

print("--- Inférence Chronos-Bolt terminée ---")

print("Nombre de prévisions :", len(chronos_q50))
print("Batch size :", BATCH_SIZE_CHRONOS)

print("\nDimensions des quantiles :", chronos_quantiles.shape)
print("Dimensions de la moyenne :", chronos_mean.shape)

print("\nNaN q10 :", np.isnan(chronos_q10).sum())
print("NaN q50 :", np.isnan(chronos_q50).sum())
print("NaN q90 :", np.isnan(chronos_q90).sum())

# Vérification de l'ordre des quantiles
quantile_crossings = np.sum(
    (chronos_q10 > chronos_q50)
    | (chronos_q50 > chronos_q90)
)

print(
    "Croisements de quantiles :",
    quantile_crossings
)

print("\nPremière prévision :")
print(f"q10 : {chronos_q10[0]:.2f} MW")
print(f"q50 : {chronos_q50[0]:.2f} MW")
print(f"q90 : {chronos_q90[0]:.2f} MW")
print(
    f"Valeur réelle : {chronos_targets[0]:.2f} MW"
)

print(f"\nDurée totale : {duree:.2f} s")
print(
    "Durée moyenne par origine : "
    f"{duree / len(chronos_q50):.4f} s"
)

--- Inférence Chronos-Bolt terminée ---
Nombre de prévisions : 7561
Batch size : 32

Dimensions des quantiles : (7561, 1, 3)
Dimensions de la moyenne : (7561, 1)

NaN q10 : 0
NaN q50 : 0
NaN q90 : 0
Croisements de quantiles : 11

Première prévision :
q10 : 1148.04 MW
q50 : 1884.77 MW
q90 : 2344.38 MW
Valeur réelle : 1993.00 MW

Durée totale : 1533.43 s
Durée moyenne par origine : 0.2028 s


In [23]:
# ===========================================================================
# Q8 — Évaluation ponctuelle de Chronos-Bolt à partir du quantile médian q50
# ===========================================================================

# Erreurs absolues de la prévision médiane
chronos_abs_errors = np.abs(
    chronos_targets - chronos_q50
)

# MAE
chronos_mae = chronos_abs_errors.mean()

# Même échelle MASE que pour la naïve, le GB et TTM
chronos_mase = chronos_mae / mase_scale

print("--- Évaluation Chronos-Bolt zero-shot ---")
print(f"Nombre de prévisions : {len(chronos_q50)}")

print(f"\nMAE Chronos-Bolt : {chronos_mae:.2f} MW")
print(
    f"Échelle saisonnière train : "
    f"{mase_scale:.2f} MW"
)
print(f"MASE Chronos-Bolt : {chronos_mase:.3f}")

--- Évaluation Chronos-Bolt zero-shot ---
Nombre de prévisions : 7561

MAE Chronos-Bolt : 741.00 MW
Échelle saisonnière train : 4996.19 MW
MASE Chronos-Bolt : 0.148


In [24]:
# ============================================================
# Q8 — Évaluation probabiliste de Chronos-Bolt
# ============================================================

import numpy as np

# ------------------------------------------------------------
# 1. Fonction de perte pinball
# ------------------------------------------------------------

def pinball_loss(y_true, y_pred, quantile):
    errors = y_true - y_pred

    return np.mean(
        np.maximum(
            quantile * errors,
            (quantile - 1) * errors
        )
    )

# ------------------------------------------------------------
# 2. Pinball loss pour q10 et q90
# ------------------------------------------------------------

pinball_q10_chronos = pinball_loss(
    chronos_targets,
    chronos_q10,
    0.10
)

pinball_q90_chronos = pinball_loss(
    chronos_targets,
    chronos_q90,
    0.90
)

pinball_mean_chronos = (
    pinball_q10_chronos + pinball_q90_chronos
) / 2

# ------------------------------------------------------------
# 3. Couverture de l'intervalle [q10, q90]
# ------------------------------------------------------------

inside_interval = (
    (chronos_targets >= chronos_q10)
    & (chronos_targets <= chronos_q90)
)

coverage_chronos = inside_interval.mean()

# Niveau nominal : 80 %
nominal_coverage = 0.80

coverage_gap = coverage_chronos - nominal_coverage

# ------------------------------------------------------------
# 4. Largeur de l'intervalle
# ------------------------------------------------------------

interval_width = chronos_q90 - chronos_q10

mean_width_chronos = interval_width.mean()
median_width_chronos = np.median(interval_width)

# ------------------------------------------------------------
# 5. Résultats
# ------------------------------------------------------------

print("--- Évaluation probabiliste Chronos-Bolt ---")

print("\nPinball loss :")
print(f"q10 : {pinball_q10_chronos:.2f} MW")
print(f"q90 : {pinball_q90_chronos:.2f} MW")
print(f"Moyenne : {pinball_mean_chronos:.2f} MW")

print("\nCouverture de l'intervalle [q10, q90] :")
print(f"Couverture observée : {coverage_chronos * 100:.2f} %")
print(f"Couverture nominale : {nominal_coverage * 100:.2f} %")
print(f"Écart : {coverage_gap * 100:+.2f} points")

print("\nLargeur de l'intervalle :")
print(f"Largeur moyenne : {mean_width_chronos:.2f} MW")
print(f"Largeur médiane : {median_width_chronos:.2f} MW")

print("\nCroisements de quantiles :")
print(f"{quantile_crossings} sur {len(chronos_targets)}")
print(
    f"Soit {100 * quantile_crossings / len(chronos_targets):.3f} %"
)

--- Évaluation probabiliste Chronos-Bolt ---

Pinball loss :
q10 : 144.56 MW
q90 : 157.85 MW
Moyenne : 151.21 MW

Couverture de l'intervalle [q10, q90] :
Couverture observée : 81.56 %
Couverture nominale : 80.00 %
Écart : +1.56 points

Largeur de l'intervalle :
Largeur moyenne : 2191.52 MW
Largeur médiane : 2118.80 MW

Croisements de quantiles :
11 sur 7561
Soit 0.145 %


## Q8 - Chronos-Bolt en zero-shot : résultats et interprétation

### Application de Chronos-Bolt

Dans cette question, on répète l'expérience de la Q7 avec un second modèle de fondation pour séries temporelles : **Chronos-Bolt**, utilisé en zero-shot avec la bibliothèque `chronos-forecasting`.

Comme pour TTM, on ne réentraîne pas le modèle sur notre série de production éolienne. On utilise directement les paramètres appris lors de son pré-entraînement et on fournit au modèle, à chaque origine de prévision, uniquement les observations historiques disponibles.

On utilise ici le modèle :

`amazon/chronos-bolt-base`

Pour chaque heure du test, on construit un contexte contenant les **2048 observations précédentes**, soit environ :

$$
\frac{2048}{24}\approx85,3\text{ jours}.
$$

On demande ensuite une prévision à **un pas**, c'est-à-dire une heure à l'avance.

Toutes les observations du test sont exploitables :

$$
\boxed{7561\text{ prévisions}}
$$

sur la même période que celle utilisée pour TTM.

### Prévisions par quantiles

Contrairement à l'évaluation ponctuelle réalisée avec TTM, Chronos-Bolt permet de récupérer directement des prévisions sous forme de quantiles.

On conserve ici :

$$
q_{0.1},\qquad q_{0.5},\qquad q_{0.9}.
$$

Le quantile médian $q_{0.5}$ est utilisé comme prévision ponctuelle pour calculer la MAE et la MASE.

Les quantiles $q_{0.1}$ et $q_{0.9}$ permettent de construire un intervalle prédictif à 80 % :

$$
[q_{0.1},q_{0.9}].
$$

### Performance ponctuelle de Chronos-Bolt

Sur les 7561 observations du test, on obtient :

$$
\boxed{MAE_{\text{Chronos-Bolt}}=741,00\text{ MW}}
$$

et, avec la même échelle saisonnière que dans les questions précédentes,

$$
\boxed{MASE_{\text{Chronos-Bolt}}=0,148}.
$$

Comme :

$$
0,148<1,221,
$$

Chronos-Bolt **bat nettement la naïve saisonnière** sur notre série.

### Comparaison des modèles

On peut maintenant comparer les deux TSFM avec la naïve saisonnière et le Gradient Boosting sur exactement les mêmes 7561 observations du test :

| Modèle | MAE sur le test (MW) | MASE sur le test |
|---|---:|---:|
| Naïve saisonnière $t-24$ | 6100,71 | 1,221 |
| **Gradient Boosting en variation** | **599,75** | **0,120** |
| Chronos-Bolt zero-shot | 741,00 | 0,148 |
| TTM zero-shot | 939,88 | 0,188 |

Le classement obtenu est donc :

$$
\boxed{
\text{Gradient Boosting}
<
\text{Chronos-Bolt}
<
\text{TTM}
<
\text{Naïve saisonnière}
}
$$

où une valeur plus faible correspond à une meilleure performance.

Les deux TSFM se comportent donc globalement de façon comparable au sens où **ils transfèrent efficacement en zero-shot et battent tous les deux largement la naïve saisonnière**.

Cependant, leurs performances ne sont pas identiques.

Chronos-Bolt obtient une MAE environ **21,2 % plus faible que TTM** :

$$
\frac{939,88-741,00}{939,88}\times100
\approx21,2\%.
$$

Sur cette série et avec notre protocole, **Chronos-Bolt est donc plus précis que TTM en zero-shot**.

En revanche, aucun des deux TSFM ne dépasse le Gradient Boosting spécialisé, dont la MASE reste la plus faible avec **0,120**.

Ce résultat montre qu'un modèle supervisé relativement classique, lorsqu'il est entraîné directement sur la série étudiée avec des variables temporelles adaptées, peut rester plus performant qu'un modèle de fondation utilisé sans adaptation.

### Évaluation probabiliste

On évalue également les quantiles $q_{0.1}$ et $q_{0.9}$ avec la **pinball loss**.

On obtient :

$$
L_{0.1}=144,56\text{ MW}
$$

et :

$$
L_{0.9}=157,85\text{ MW}.
$$

La perte pinball moyenne vaut donc :

$$
\boxed{151,21\text{ MW}}.
$$

Pour l'intervalle :

$$
[q_{0.1},q_{0.9}],
$$

le niveau nominal attendu est de 80 %.

La couverture observée est :

$$
\boxed{81,56\%},
$$

soit un écart de seulement :

$$
\boxed{+1,56\text{ point}}
$$

par rapport aux 80 % attendus.

L'intervalle est donc légèrement plus couvrant que le niveau nominal, tout en restant assez proche de celui-ci.

Sa largeur moyenne vaut :

$$
\boxed{2191,52\text{ MW}},
$$

et sa largeur médiane :

$$
\boxed{2118,80\text{ MW}}.
$$

La couverture ne doit pas être interprétée seule : un intervalle très large pourrait obtenir une bonne couverture tout en étant peu informatif. On regarde donc conjointement la couverture, la largeur et les pertes pinball.

### Croisements de quantiles

On observe également :
$$
\boxed{11\text{ croisements de quantiles sur }7561}
$$

soit environ :

$$
\boxed{0,145\%}.
$$

Pour ces quelques observations, l'ordre attendu

$$
q_{0.1}\leq q_{0.5}\leq q_{0.9}
$$

n'est pas respecté.

Le phénomène reste très minoritaire, mais on le conserve et on le signale explicitement. On ne réordonne pas les quantiles après avoir observé le test, afin de ne pas modifier a posteriori les sorties du modèle.

### Temps de calcul

L'inférence complète de Chronos-Bolt sur les 7561 origines a pris environ :

$$
1333,44\text{ s}\approx22,2\text{ minutes}
$$

sur CPU.

Dans notre environnement, Chronos-Bolt demande donc nettement plus de temps d'inférence que TTM. Cependant, ce résultat dépend du matériel, de l'implémentation, de la taille des modèles et du traitement par batch. On ne l'interprète donc pas comme une conclusion générale sur la vitesse relative des deux familles de modèles.

### Interprétation

Les deux TSFM donnent des résultats convaincants en zero-shot :

$$
MASE_{TTM}=0,188
$$

et :

$$
MASE_{\text{Chronos-Bolt}}=0,148.
$$

Ils battent tous les deux très largement la naïve saisonnière :

$$
MASE_{\text{naïve}}=1,221.
$$

Chronos-Bolt obtient cependant une meilleure performance ponctuelle que TTM sur cette série.

Ce résultat montre aussi qu'il ne suffit pas de dire qu'un modèle appartient à la famille des TSFM pour prévoir sa performance. Deux modèles pré-entraînés peuvent avoir des architectures, des stratégies de pré-entraînement et des mécanismes de prévision différents, ce qui peut conduire à des résultats différents sur une même série.

Dans notre expérience, les deux modèles montrent une bonne capacité de transfert en zero-shot, mais **Chronos-Bolt s'adapte mieux à cette série selon la MASE obtenue**.

### Limites et précautions

On ne teste ici qu'une configuration de Chronos-Bolt et une configuration de TTM. On n'a pas cherché plusieurs modèles ou plusieurs longueurs de contexte sur le jeu de test pour retenir a posteriori la meilleure configuration.

Le résultat concerne également une seule série de production éolienne et un horizon de prévision d'une heure. On ne peut donc pas conclure que Chronos-Bolt sera systématiquement supérieur à TTM sur d'autres séries ou d'autres horizons.

Enfin, les 11 croisements de quantiles montrent que les sorties probabilistes ne respectent pas toujours parfaitement l'ordre théorique des quantiles. Ce phénomène est faible dans notre expérience, mais il doit être signalé.

### Conclusion

Sur notre série de production éolienne, les deux TSFM testés en zero-shot battent clairement la naïve saisonnière.

On obtient :

$$
\boxed{
MASE_{GB}=0,120
<
MASE_{Chronos}=0,148
<
MASE_{TTM}=0,188
<
MASE_{\text{naïve}}=1,221
}
$$

Chronos-Bolt est donc le meilleur des deux TSFM dans notre expérience, mais le Gradient Boosting spécialisé reste le modèle le plus précis.

Le résultat est néanmoins important : **sans réentraînement spécifique sur notre série, TTM et Chronos-Bolt obtiennent tous les deux des performances nettement supérieures à la référence naïve**.

On conserve donc le résultat tel qu'il est, sans chercher à favoriser artificiellement un modèle. Dans notre cas, il n'y a pas de résultat négatif à signaler vis-à-vis de la naïve : **les deux TSFM la battent clairement**.

# Synthèse générale (Partie B) : Prévision zero-shot avec des TSFM

## Objectif de cette partie

Dans cette partie, on a étudié une approche différente des modèles supervisés classiques utilisés dans la Partie A.

L'objectif était d'évaluer des **Time Series Foundation Models (TSFM)** déjà pré-entraînés sur de grands ensembles de séries temporelles et de vérifier s'ils pouvaient être appliqués directement à notre série de production éolienne allemande, **sans réentraînement spécifique**.

On a testé deux modèles :

- **Tiny Time Mixers (TTM)** ;
- **Chronos-Bolt**.

Les deux modèles ont été utilisés en **zero-shot** : leurs paramètres pré-entraînés sont conservés et aucune phase d'apprentissage supplémentaire n'est réalisée sur notre série cible.

L'intérêt est donc de mesurer leur capacité de **transfert** vers une série qu'ils n'ont pas été spécifiquement entraînés à prévoir dans notre expérience.

## Protocole expérimental commun

Pour rendre la comparaison aussi cohérente que possible avec la Partie A, on conserve le même jeu de test chronologique :

$$
7561\text{ observations}
$$

comprises entre :

$$
\text{20 novembre 2019 à 23h}
$$

et :

$$
\text{30 septembre 2020 à 23h}.
$$

À chaque origine de prévision, le modèle reçoit uniquement les observations disponibles dans le passé.

On réalise une prévision à :

$$
h=1,
$$

c'est-à-dire **une heure à l'avance**.

Cette procédure reproduit une situation de prévision à origine glissante et évite d'utiliser des observations futures pour construire les entrées du modèle.

## Choix des contextes

Les deux TSFM n'utilisent pas exactement la même longueur de contexte.

Pour TTM, on utilise :
$$
1536\text{ observations},
$$

soit :

$$
\frac{1536}{24}=64\text{ jours}.
$$

Pour Chronos-Bolt, on utilise :

$$
2048\text{ observations},
$$

soit environ :

$$
\frac{2048}{24}\approx85,3\text{ jours}.
$$

Cette différence provient des configurations retenues pour chaque modèle.

On n'a pas testé plusieurs longueurs de contexte sur le jeu de test afin de sélectionner celle donnant le meilleur résultat. Cela permet d'éviter d'utiliser indirectement le test comme jeu de validation.

## TTM en zero-shot

TTM est appliqué directement à chaque contexte historique sans fine-tuning.

Sur les 7561 observations du test, on obtient :
$$
MAE_{TTM}=939,88\text{ MW}.
$$

Avec la même échelle saisonnière calculée sur le jeu d'entraînement :
$$
4996,19\text{ MW},
$$

on obtient :
$$
\boxed{MASE_{TTM}=0,188}.
$$

Comme cette valeur est très inférieure à 1, TTM fait nettement mieux que la référence saisonnière utilisée pour définir la MASE.

Ce résultat est particulièrement intéressant car le modèle n'a pas été réentraîné sur notre série.

## Chronos-Bolt en zero-shot

On répète ensuite le protocole avec Chronos-Bolt.

Chronos-Bolt fournit des prévisions sous forme de quantiles. On récupère :
$$
q_{0.1},\qquad q_{0.5},\qquad q_{0.9}.
$$

Pour la comparaison ponctuelle, on utilise le quantile médian :
$$
q_{0.5}.
$$

On obtient :

$$
MAE_{Chronos}=741,00\text{ MW}
$$

et :

$$
\boxed{MASE_{Chronos}=0,148}.
$$

Chronos-Bolt bat donc lui aussi très largement la naïve saisonnière.

Il obtient également une MAE environ **21,2 % plus faible que TTM** sur cette série.

## Comparaison globale des modèles

En réunissant les résultats obtenus sur le même jeu de test, on obtient :

| Modèle | MAE test (MW) | MASE test |
|---|---:|---:|
| **Gradient Boosting en variation** | **599,75** | **0,120** |
| **Chronos-Bolt zero-shot** | **741,00** | **0,148** |
| **TTM zero-shot** | **939,88** | **0,188** |
| **Naïve saisonnière $t-24$** | **6100,71** | **1,221** |

Le classement obtenu est donc :

$$
\boxed{
GB < Chronos\text{-}Bolt < TTM < Naïve
}
$$

pour la MAE et la MASE, puisqu'une valeur plus faible indique une meilleure performance.

## Ce que cette comparaison montre

Le premier résultat important est que **les deux TSFM battent largement la naïve saisonnière**.

On obtient :
$$
MASE_{TTM}=0,188
$$

et :
$$
MASE_{Chronos}=0,148
$$

contre :

$$
MASE_{Naïve}=1,221.
$$

Les modèles pré-entraînés montrent donc une bonne capacité de transfert vers notre série, même sans adaptation spécifique.

Le deuxième résultat est que les deux TSFM ne sont pas équivalents.

Dans notre expérience :
$$
MASE_{Chronos}<MASE_{TTM}.
$$

Chronos-Bolt est donc plus précis que TTM sur cette série et pour cet horizon.

Cependant, ce résultat ne permet pas de conclure que Chronos-Bolt est systématiquement supérieur à TTM. Il concerne une série particulière, une période particulière et un horizon de prévision d'une heure.

## Un résultat important : le Gradient Boosting reste meilleur

Le résultat probablement le plus intéressant de cette partie est que le meilleur modèle ponctuel reste le **Gradient Boosting entraîné spécifiquement sur notre série** :
$$
MASE_{GB}=0,120.
$$

Il devance :

$$
MASE_{Chronos}=0,148
$$

et :

$$
MASE_{TTM}=0,188.
$$

Cela montre qu'un modèle de fondation n'est pas automatiquement supérieur à un modèle supervisé plus classique.

Le Gradient Boosting bénéficie d'un apprentissage directement adapté à notre série et de variables construites spécifiquement pour le problème, notamment les retards temporels.

À l'inverse, les TSFM sont ici utilisés sans adaptation.

La comparaison met donc en évidence deux logiques différentes :

- **spécialisation**, pour le Gradient Boosting ;
- **capacité de transfert**, pour les TSFM.

Le fait que les TSFM se rapprochent du modèle spécialisé sans être réentraînés constitue déjà un résultat intéressant.

## Apport probabiliste de Chronos-Bolt

Chronos-Bolt apporte également une dimension probabiliste grâce à ses quantiles.

Pour l'intervalle :
$$
[q_{0.1},q_{0.9}],
$$

le niveau nominal est de :

$$
80\%.
$$

On obtient une couverture observée de :

$$
\boxed{81,56\%}.
$$

L'écart au niveau nominal est donc :

$$
\boxed{+1,56\text{ point}}.
$$

La largeur moyenne de l'intervalle vaut :

$$
2191,52\text{ MW},
$$

et sa largeur médiane :

$$
2118,80\text{ MW}.
$$

Les pertes pinball sont :

$$
L_{0.1}=144,56\text{ MW},
$$

$$
L_{0.9}=157,85\text{ MW},
$$

avec une moyenne de :

$$
\boxed{151,21\text{ MW}}.
$$

La couverture est donc assez proche des 80 % attendus.

Il faut néanmoins toujours interpréter la couverture conjointement avec la largeur des intervalles : obtenir une couverture élevée avec des intervalles extrêmement larges ne constituerait pas nécessairement une bonne prévision probabiliste.

## Croisements de quantiles

On observe : $11$, croisements de quantiles sur : 7561
prévisions, soit :
$$
\boxed{0,145\%}.
$$

Dans ces quelques cas, l'ordre théorique :

$$
q_{0.1}\leq q_{0.5}\leq q_{0.9}
$$

n'est pas respecté.

Le phénomène est très minoritaire, mais on le signale explicitement.

On choisit de conserver les sorties brutes du modèle plutôt que de réordonner les quantiles après observation du jeu de test.

## Temps de calcul observés

Dans notre environnement CPU, les deux modèles présentent également des coûts d'inférence très différents.

Pour TTM, l'inférence complète sur les 7561 origines a pris environ :
$$
7,24\text{ s}
$$

Pour Chronos-Bolt, elle a pris :

$$
1533,43\text{ s}
$$

soit environ :

$$
25,3\text{ minutes}.
$$

Dans notre environnement expérimental, TTM est donc beaucoup plus rapide.

Cependant, cette différence ne doit pas être interprétée comme une propriété universelle des deux modèles.

Le temps d'exécution dépend notamment :

- du matériel ;
- de la taille et de l'architecture du modèle ;
- de son implémentation ;
- de la taille des batches ;
- des optimisations disponibles ;
- de la longueur du contexte.

On considère donc ces temps comme des **mesures propres à notre expérience**, et non comme un benchmark général.

## Choix méthodologiques importants

Plusieurs décisions ont été prises afin de conserver un protocole rigoureux.

**Premièrement**, on conserve exactement la séparation chronologique définie dans la Partie A.

**Deuxièmement**, les TSFM sont utilisés sans réentraînement afin d'évaluer réellement leur capacité zero-shot.

**Troisièmement**, on ne teste pas plusieurs configurations sur le jeu de test pour sélectionner ensuite celle qui donne le meilleur résultat.

**Quatrièmement**, on utilise la même échelle MASE issue du jeu d'entraînement :$4996,19\text{ MW}$, ce qui rend les MASE directement comparables.

**Cinquièmement**, on conserve les résultats tels qu'ils sont obtenus, y compris les croisements de quantiles observés avec Chronos-Bolt.

L'objectif n'est donc pas de chercher artificiellement le meilleur score, mais d'obtenir une comparaison reproductible et méthodologiquement cohérente.

## Limites de cette partie

Cette étude présente plusieurs limites.

On travaille sur une seule variable de production éolienne allemande.

On étudie uniquement un horizon : $h=1\text{ heure}$. Les conclusions pourraient être différentes pour des horizons plus longs.

Les deux TSFM n'utilisent pas exactement la même longueur de contexte.

On n'effectue pas ici de fine-tuning des modèles de fondation.

Enfin, une bonne performance sur cette série ne permet pas de conclure qu'un modèle sera systématiquement meilleur sur d'autres séries énergétiques.

Ces limites ouvrent naturellement la voie à la suite du projet, où on pourra étudier l'adaptation des modèles et leur comportement sur d'autres configurations.

## Ce qu'on retient de la Partie B

Cette partie montre trois résultats principaux.

**1. Les TSFM peuvent être efficaces sans réentraînement.**

TTM et Chronos-Bolt battent tous les deux très largement la naïve saisonnière en zero-shot.

**2. Tous les TSFM ne donnent pas la même performance.**

Sur notre série : $Chronos\text{-}Bolt > TTM$ en termes de précision, puisque sa MASE est plus faible.

**3. Un modèle spécialisé peut encore être meilleur.**

Le Gradient Boosting reste devant les deux modèles de fondation :

$$
\boxed{
0,120 < 0,148 < 0,188 < 1,221
}
$$

pour :

$$
GB,\ Chronos\text{-}Bolt,\ TTM,\ Naïve.
$$

La conclusion n'est donc pas que les modèles de fondation remplacent automatiquement les approches classiques.

Le résultat intéressant est plutôt qu'ils sont capables d'obtenir **de bonnes performances immédiatement, sans apprentissage spécifique sur la nouvelle série**.

# Références et ressources
1. Ekambaram, V., et al. (2024).
   Tiny Time Mixers (TTMs): Fast pre-trained models
   for enhanced zero/few-shot forecasting of multivariate
   time series.
   https://arxiv.org/abs/2401.03955

2. Ansari, A. F., et al. (2024).
   Chronos: Learning the language of time series.
   https://arxiv.org/abs/2403.07815

3. Rasul, K., et al. (2024).
   Lag-Llama: Towards foundation models for
   probabilistic time series forecasting.
   https://arxiv.org/abs/2310.08278

4. Noble, J.
   *Using the IBM Granite models for time series forecasting.*
   IBM Developer.
   https://developer.ibm.com/tutorials/awb-foundation-model-time-series-forecasting/

5. Woo, G., et al. (2024).
   Unified training of universal time series forecasting
   transformers.
   https://arxiv.org/abs/2402.02532

6. Garza, A., & Mergenthaler-Canseco, M. (2023).
   TimeGPT-1.
   https://arxiv.org/abs/2310.03589